# ARGUS - AutoRegressive Generative User Sequential modeling

Implementation based on: [Scaling Recommender Transformers to One Billion Parameters](https://arxiv.org/abs/2507.15994) (Yandex, 2025)


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import os

import numpy as np
import pandas as pd

os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
# os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'

device = 'cpu'
print(f"Using device: {device}")

torch.set_num_threads(1)
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

print(f"Torch threads: {torch.get_num_threads()}")


import sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

Using device: cpu
Torch threads: 1


## Сбор фичей для Argus

In [ ]:
import os
import polars as pl
import numpy as np
import gc

def load_tecd_data_with_mappings(
    data_path='t_ecd_small_partial/dataset/small',
    domain='retail',
    day_begin=1082,
    day_end=1308,
    filter_file='processed_data/train.parquet',
    output_path='processed_data/argus_full_data.parquet'
):
    from tecd_retail_recsys.data.preprocess import DataPreprocessor
    
    print("Loading preprocessor and creating mappings...")
    preprocessor = DataPreprocessor(
        raw_data_path=data_path,
        day_begin=day_begin,
        day_end=day_end,
        min_user_interactions=1,
        min_item_interactions=20
    )
    
    raw_data = preprocessor.load_raw_data()
    items_df = preprocessor.load_items_data()
    raw_data = preprocessor.merge_items_features(raw_data, items_df)
    raw_data = preprocessor.filter_events(raw_data)
    raw_data = preprocessor.filter_by_interactions(raw_data)
    preprocessor.create_mappings(raw_data)
    
    train_indices = pl.read_parquet(filter_file)
    valid_user_indices = train_indices['user_id'].unique().to_list()
    valid_item_indices = train_indices['item_id'].unique().to_list()
    del train_indices
    gc.collect()
    
    valid_user_ids = [preprocessor.idx_to_user[idx] for idx in valid_user_indices]
    valid_item_ids = [preprocessor.idx_to_item[idx] for idx in valid_item_indices]
    
    print(f"Filter: {len(valid_user_ids):,} users, {len(valid_item_ids):,} items")
    
    events_dir = os.path.join(data_path, domain, 'events')
    events = pl.concat([
        pl.scan_parquet(os.path.join(events_dir, f'0{day}.pq')).with_columns(pl.lit(day).alias('day'))
        for day in range(day_begin, day_end + 1)
        if os.path.exists(os.path.join(events_dir, f'0{day}.pq'))
    ]).filter(
        pl.col('user_id').is_in(valid_user_ids) & 
        pl.col('item_id').is_in(valid_item_ids)
    )
    
    items = pl.scan_parquet(
        os.path.join(data_path, domain, 'items.pq')
    ).filter(pl.col('item_id').is_in(valid_item_ids)).rename({
        'brand_id': 'item_brand_id',
        'category': 'item_category',
        'subcategory': 'item_subcategory',
        'price': 'item_price',
        'embedding': 'item_embedding'
    })
    
    users = pl.scan_parquet(
        os.path.join(data_path, 'users.pq')
    ).filter(pl.col('user_id').is_in(valid_user_ids)).rename({
        'socdem_cluster': 'user_socdem_cluster',
        'region': 'user_region'
    })
    
    data = (events
            .join(items, on='item_id', how='left')
            .join(users, on='user_id', how='left'))
    
    try:
        brands = pl.read_parquet(
            os.path.join(data_path, 'brands.pq'),
            use_pyarrow=False
        ).rename({
            'brand_id': 'item_brand_id',
            'embedding': 'brand_embedding'
        })
        
        brands = (brands
                  .sort('brand_embedding', nulls_last=True)
                  .unique(subset=['item_brand_id'], keep='first'))
        
        print(f"Loaded {len(brands):,} unique brands (prioritized non-null embeddings)")
        
        data = data.join(
            brands.lazy().select(['item_brand_id', 'brand_embedding']),
            on='item_brand_id',
            how='left'
        )

        del brands
        gc.collect()

    except Exception as e:
        print(f"Warning: Could not load brands.pq ({e}), brand_embedding will be None")
        data = data.with_columns(pl.lit(None).alias('brand_embedding'))
    
    data = data.with_columns(
        (pl.col('timestamp').dt.total_seconds()).cast(pl.Int64).alias('timestamp')
    )
    
    print("\nApplying mappings to convert IDs to indices...")
    
    user_mapping = pl.DataFrame({
        'user_id_orig': list(preprocessor.user_to_idx.keys()),
        'user_id_mapped': list(preprocessor.user_to_idx.values())
    })
    
    item_mapping = pl.DataFrame({
        'item_id_orig': list(preprocessor.item_to_idx.keys()),
        'item_id_mapped': list(preprocessor.item_to_idx.values())
    })
    
    data = (data
            .join(user_mapping.lazy(), left_on='user_id', right_on='user_id_orig', how='left')
            .drop('user_id')
            .rename({'user_id_mapped': 'user_id'}))
    
    data = (data
            .join(item_mapping.lazy(), left_on='item_id', right_on='item_id_orig', how='left')
            .drop('item_id')
            .rename({'item_id_mapped': 'item_id'}))
    
    # Конвертация эмбеддингов в строковый формат для экономии памяти
    # Для конвертации Array в String используем map_batches
    # data = data.with_columns([
    #     pl.col('item_embedding').map_elements(lambda x: str(x.to_list()) if x is not None else None, return_dtype=pl.String).alias('item_embedding'),
    #     pl.col('brand_embedding').map_elements(lambda x: str(x.to_list()) if x is not None else None, return_dtype=pl.String).alias('brand_embedding')
    # ])
    # пока просто удалю их для экономии памяти
    data = data.drop(['item_embedding', 'brand_embedding'])


    print("Final deduplication...")
    data = data.unique()
    
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    print(f"\nSaving to {output_path}...")
    data.sink_parquet(output_path, compression='snappy')
    print(f"Done!")
    
    stats = data.select([
        pl.count().alias('total_events'),
        pl.col('user_id').n_unique().alias('unique_users'),
        pl.col('item_id').n_unique().alias('unique_items')
    ]).collect()
    
    print(f"\nFiltered data stats (after deduplication):")
    print(f"  Events: {stats['total_events'][0]:,}")
    print(f"  Users: {stats['unique_users'][0]:,} (indices: 0-{stats['unique_users'][0]-1})")
    print(f"  Items: {stats['unique_items'][0]:,} (indices: 0-{stats['unique_items'][0]-1})")
    
    sample_df = data.fetch(10000)
    sample_memory_mb = sample_df.estimated_size('mb')
    estimated_total_mb = (sample_memory_mb / len(sample_df)) * stats['total_events'][0]
    estimated_total_gb = estimated_total_mb / 1024
    
    print(f"\nEstimated memory: {estimated_total_mb:.2f} MB ({estimated_total_gb:.2f} GB)")
    
    del sample_df
    gc.collect()

    return output_path

output_file = load_tecd_data_with_mappings()

Loading preprocessor and creating mappings...
Loading events from t_ecd_small_partial/dataset/small/retail/events
Loaded 236,479,226 total events
Loading items data from t_ecd_small_partial/dataset/small/retail/items.pq
Loaded 250,171 items with features: ['item_id', 'item_brand_id', 'item_category', 'item_subcategory', 'item_price', 'item_embedding']
Merged item features. Data shape: (236479226, 12)
Filtered to 3,758,762 events with action_type='added-to-cart'
After filtering (min_user_interactions=1, min_item_interactions=20): 3,249,972 events, 84,944 users, 30,954 items
Created mappings: 84944 users, 30954 items
Filter: 7,415 users, 30,354 items
Loaded 24,467 unique brands (prioritized non-null embeddings)

Applying mappings to convert IDs to indices...
Final deduplication...

Saving to processed_data/argus_data_filtered.parquet...
Done!

Filtered data stats (after deduplication):
  Events: 14,031,889
  Users: 7,412 (indices: 0-7411)
  Items: 30,345 (indices: 0-30344)

Estimated mem

In [79]:
data = pd.read_parquet('processed_data/argus_full_data.parquet')
data['item_category'] = data['item_category'].fillna('unkown')
data['item_subcategory'] = data['item_subcategory'].fillna('unkown')
data.head()

,timestamp,subdomain,action_type,os,day,item_brand_id,item_category,item_subcategory,item_price,user_socdem_cluster,user_region,user_id,item_id
0,107565387,catalog,view,android,1244,37799,Foodstuffs and Beverages,Sweet Desserts and Confectionery,-4.767958,9,60.0,11409,15379
1,113092466,cart,click,ios,1308,60434,Foodstuffs and Beverages,Sweet Desserts and Confectionery,-4.580000,9,2.0,27977,15535
2,113093139,catalog,view,android,1308,37799,Foodstuffs and Beverages,Sweet Desserts and Confectionery,-4.299029,4,59.0,1916,16222
3,113094199,catalog,view,android,1308,146468,Foodstuffs and Beverages,Ready-to-Eat Food Products,-4.080918,4,59.0,1916,26888
4,113092605,catalog,view,ios,1308,60434,Foodstuffs and Beverages,Dairy Products and Eggs,-4.770505,9,2.0,27977,11036


In [82]:
train_df = data[data['day'] < 1269]
val_df = data[(data['day'] >= 1269) & (data['day'] < 1288)]
test_df = data[data['day'] >= 1289]

print(train_df.shape, val_df.shape, test_df.shape)

# пропуски в ценах
train_med_price = train_df['item_price'].median()
train_df['item_price'] = train_df['item_price'].fillna(train_med_price)
val_df['item_price'] = val_df['item_price'].fillna(train_med_price)
test_df['item_price'] = test_df['item_price'].fillna(train_med_price)

# пропуски в регионе
train_popular_region = train_df['user_region'].mode()[0]
train_df['user_region'] = train_df['user_region'].fillna(train_popular_region)
val_df['user_region'] = val_df['user_region'].fillna(train_popular_region)
test_df['user_region'] = test_df['user_region'].fillna(train_popular_region)

train_df.to_parquet('processed_data/argus_train_data.parquet')
val_df.to_parquet('processed_data/argus_val_data.parquet')
test_df.to_parquet('processed_data/argus_test_data.parquet')

(10231576, 13) (1872284, 13) (1765876, 13)


## Обучение и инференс ARGUS

In [3]:
from tecd_retail_recsys.models.argus import (
    FullARGUSConfig,
    ArgusDataPreprocessor,
    FullARGUSTrainer,
    train_val_split
)

In [ ]:
train_df = pd.read_parquet('processed_data/argus_train_data.parquet')

# если оставляю только add_to_cart события
# train_df = train_df[train_df['action_type'] == 'added-to-cart']

raw_records = train_df.to_dict('records')

In [ ]:
cfg = FullARGUSConfig(
    # размерности предобученных эмбеддингов
    item_emb_dim=128,
    brand_emb_dim=64,

    # архитектура
    d_model=64,
    n_heads=4,
    n_layers=4,
    d_ff=512,
    dropout=0.2,
    max_interactions=150,

    # лосс
    n_uniform_negatives=8192,
    use_in_batch_negatives=True,
    temperature=1,
    feedback_loss_weight=0.1,
    label_smoothing=0.1,

    # обучение
    lr=1e-3,
    weight_decay=1e-2,
    warmup_steps=500,
    epochs=3,
    batch_size=64,

    top_k=100,
    device='mps'
)


In [ ]:
preprocessor = ArgusDataPreprocessor(cfg)
preprocessor.fit(raw_records)
user_sequences = preprocessor.transform(raw_records)

print(f"Пользователей с историей ≥ 1: {len(user_sequences)}")


Пользователей с историей ≥ 1: 6416


In [ ]:
train_seqs, val_seqs = train_val_split(user_sequences, n_val_items=1)
print(f"Train users: {len(train_seqs)}, Val users: {len(val_seqs)}")

Train users: 6416, Val users: 5809


In [ ]:
trainer = FullARGUSTrainer(cfg, preprocessor, train_seqs, val_seqs)
trainer.train()

[ARGUS] Каталог: 30214 items, 7 brands, 20 categories, 3 action types
[ARGUS] Параметров модели: 4,251,076


100%|██████████| 2186/2186 [19:40<00:00,  1.85it/s]


Epoch 1/3 | item_loss=8.2845 | fb_loss=0.4182 | lr=8.21e-04 | NDCG@100=0.0356 ★


100%|██████████| 2186/2186 [19:05<00:00,  1.91it/s]


Epoch 2/3 | item_loss=7.5522 | fb_loss=0.4056 | lr=2.88e-04 | NDCG@100=0.0388 ★


100%|██████████| 2186/2186 [20:03<00:00,  1.82it/s]


Epoch 3/3 | item_loss=7.4184 | fb_loss=0.4039 | lr=4.07e-09 | NDCG@100=0.0409 ★

[ARGUS] Обучение завершено. Best NDCG@100 = 0.0409


In [51]:
torch.save({
    'model_state': trainer.model.state_dict(),
    'cfg': trainer.cfg,
    'preprocessor': trainer.preprocessor,
}, 'models/argus/exp13_model.pt')


In [ ]:
# выбираем пользователей для рекомендаций
users_to_recommend = {
    uid: seq for uid, seq in list(user_sequences.items())
}

In [ ]:
# возвращает item_id + скоры
recommendations = trainer.recommend(users_to_recommend, top_k=100)
for uid, recs in list(recommendations.items())[:3]:
    print(f"\nUser {uid}:")
    print(f"  Top-5: {recs[:5]}")


User 11409:
  Top-5: [(16066, 3.5642287731170654), (26537, 3.5567193031311035), (1683, 3.5202112197875977), (20587, 3.510333776473999), (21652, 3.4786148071289062)]

User 2981:
  Top-5: [(8589, 2.9788854122161865), (11413, 2.713728904724121), (19066, 2.648096799850464), (28042, 2.554144859313965), (8476, 2.548130989074707)]

User 58721:
  Top-5: [(16405, 6.7764153480529785), (13020, 6.454838752746582), (27827, 6.0004143714904785), (8589, 5.974541664123535), (12053, 5.905001640319824)]


In [ ]:
# просто список item_id
simple_recs = trainer.recommend_simple(users_to_recommend, top_k=100)

In [ ]:
from tecd_retail_recsys.data.preprocess import DataPreprocessor
dp = DataPreprocessor(day_begin=1082, day_end=1308, val_days=20, test_days=20, min_user_interactions=1, min_item_interactions=20)
train_df, val_df, test_df = dp.preprocess()
joined = dp.get_grouped_data(train_df, val_df, test_df)
joined['train_val_interactions'] = joined['train_interactions'] + joined['val_interactions']

In [57]:
argus_recs_df = pd.DataFrame(simple_recs.items(), columns=['user_id', 'argus_recs'])
argus_recs_df = argus_recs_df.merge(joined, on='user_id')

In [58]:
from tecd_retail_recsys.metrics import calculate_metrics
argus_metrics_toppop = calculate_metrics(argus_recs_df, train_col='train_interactions', model_preds='argus_recs', gt_col='val_interactions', verbose=True)

[Metrics debug] resolved gt_col='val_interactions' item_id_index=0
[Metrics debug] ratings_true shape: (19604, 3) ratings_pred shape: (64300, 3)
  ratings_true dtypes: {'user_id': dtype('int64'), 'item_id': dtype('int64')}
  ratings_pred dtypes: {'user_id': dtype('int64'), 'item_id': dtype('int64')}
  user_id=11409 gt_count=27 pred_count=100 overlap=6
  user_id=2930 gt_count=54 pred_count=100 overlap=0
    [ID spaces] gt sample=[2186, 2325, 2439, 2848, 3535] range=[2186, 30825] | rec sample=[368, 1975, 2055, 2226, 2386] range=[368, 30244]
  user_id=38220 gt_count=19 pred_count=100 overlap=1

At k=10:
  MAP@10       = 0.0533
  NDCG@10      = 0.1458
  Precision@10 = 0.0838
  Recall@10    = 0.0262

At k=100:
  MAP@100       = 0.0218
  NDCG@100      = 0.1098
  Precision@100 = 0.0302
  Recall@100    = 0.1000

Other Metrics:
  MRR                 = 0.1850
  Catalog Coverage    = 0.2175
  Diversity     = 0.9764  [0=same recs for all, 1=unique recs]
  Novelty             = 0.7986
  Serendipity

### Инференс лучшей модели

In [65]:
checkpoint = torch.load('models/argus/exp12_model.pt', weights_only=False)
cfg = checkpoint['cfg']
preprocessor = checkpoint['preprocessor']

trainer = FullARGUSTrainer(cfg, preprocessor, train_seqs, val_seqs)
trainer.model.load_state_dict(checkpoint['model_state'])
trainer.model.eval()

[ARGUS] Каталог: 26926 items, 7 brands, 20 categories, 1 action types
[ARGUS] Параметров модели: 3,829,826


FullARGUS(
  (context_encoder): FeatureEncoder(
    (cat_embeddings): ModuleDict(
      (socdem_cluster): Embedding(24, 64, padding_idx=0)
      (region): Embedding(93, 64, padding_idx=0)
      (os_id): Embedding(3, 64, padding_idx=0)
      (subdomain_id): Embedding(6, 64, padding_idx=0)
      (hour): Embedding(26, 64, padding_idx=0)
      (dow): Embedding(9, 64, padding_idx=0)
      (time_delta_bucket): Embedding(66, 64, padding_idx=0)
    )
    (dense_projections): ModuleDict()
  )
  (item_encoder): FeatureEncoder(
    (cat_embeddings): ModuleDict(
      (item_id): Embedding(26927, 64, padding_idx=0)
      (brand_id): Embedding(8, 64, padding_idx=0)
      (category_id): Embedding(21, 64, padding_idx=0)
      (subcategory_id): Embedding(82, 64, padding_idx=0)
      (price_bucket): Embedding(102, 64, padding_idx=0)
    )
    (dense_projections): ModuleDict()
  )
  (feedback_encoder): FeatureEncoder(
    (cat_embeddings): ModuleDict(
      (action_type): Embedding(2, 64, padding_idx=0)


In [66]:
train_df = pd.read_parquet('processed_data/argus_train_data.parquet')
train_df = train_df[train_df['action_type'] == 'added-to-cart']
raw_records = train_df.to_dict('records')
user_sequences = preprocessor.transform(raw_records)

users_to_recommend = {
    uid: seq for uid, seq in list(user_sequences.items())
}

In [67]:
simple_recs = trainer.recommend_simple(users_to_recommend, top_k=100)
argus_recs_df = pd.DataFrame(simple_recs.items(), columns=['user_id', 'argus_recs'])
argus_recs_df = argus_recs_df.merge(joined, on='user_id')

In [68]:
argus_best_metrics = calculate_metrics(argus_recs_df, train_col='train_interactions', model_preds='argus_recs', gt_col='val_interactions', verbose=True)

[Metrics debug] resolved gt_col='val_interactions' item_id_index=0
[Metrics debug] ratings_true shape: (19516, 3) ratings_pred shape: (64000, 3)
  ratings_true dtypes: {'user_id': dtype('int64'), 'item_id': dtype('int64')}
  ratings_pred dtypes: {'user_id': dtype('int64'), 'item_id': dtype('int64')}
  user_id=27464 gt_count=63 pred_count=100 overlap=7
  user_id=2577 gt_count=28 pred_count=100 overlap=2
  user_id=47956 gt_count=15 pred_count=100 overlap=0
    [ID spaces] gt sample=[5631, 6134, 7982, 11488, 11940] range=[5631, 30950] | rec sample=[302, 368, 1133, 1601, 1634] range=[302, 30316]

At k=10:
  MAP@10       = 0.0823
  NDCG@10      = 0.2033
  Precision@10 = 0.1047
  Recall@10    = 0.0306

At k=100:
  MAP@100       = 0.0292
  NDCG@100      = 0.1268
  Precision@100 = 0.0345
  Recall@100    = 0.1027

Other Metrics:
  MRR                 = 0.1903
  Catalog Coverage    = 0.0950
  Diversity     = 0.9459  [0=same recs for all, 1=unique recs]
  Novelty             = 0.8158
  Serendipit

In [71]:
from tecd_retail_recsys.utils import get_avg_recs_price, get_item_to_price
item_to_price = get_item_to_price(dp)

avg_argus_recs_price_val = get_avg_recs_price(argus_recs_df, item_to_price, 'argus_recs')
print(f'Средняя цена рекомендаций модели ARGUS на валидации: {avg_argus_recs_price_val:.2f}')

Средняя цена рекомендаций модели ARGUS на валидации: -3.91


### **ARGUS: эксперименты**

<table border="1" cellpadding="6" cellspacing="0" style="border-collapse: collapse; font-size: 14px;">
  <thead>
    <tr style="background-color: #f0f0f0;">
      <th>#</th>
      <th>d_model</th>
      <th>n_layers</th>
      <th>d_ff</th>
      <th>dropout</th>
      <th>negatives</th>
      <th>temperature</th>
      <th>fb_weight</th>
      <th>label_smooth</th>
      <th>lr</th>
      <th>epochs</th>
      <th>loss_type</th>
      <th>data</th>
      <th>augmentation</th>
      <th>NDCG val</th>
      <th>Изменение</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>1</td>
      <td>128</td>
      <td>4</td>
      <td>512</td>
      <td>0.1</td>
      <td>2048</td>
      <td>0.05</td>
      <td>0.3</td>
      <td>0</td>
      <td>4e-4</td>
      <td>30</td>
      <td>sampled</td>
      <td>all</td>
      <td>—</td>
      <td>0.0281</td>
      <td>бейзлайн</td>
    </tr>
    <tr>
      <td>2</td>
      <td>64</td>
      <td>2</td>
      <td>512</td>
      <td>0.1</td>
      <td>2048</td>
      <td>0.05</td>
      <td>0.3</td>
      <td>0</td>
      <td>4e-4</td>
      <td>10</td>
      <td>sampled</td>
      <td>all</td>
      <td>—</td>
      <td>0.0209</td>
      <td>уменьшил модель</td>
    </tr>
    <tr>
      <td>3</td>
      <td>64</td>
      <td>2</td>
      <td>512</td>
      <td>0.1</td>
      <td>8192</td>
      <td>0.05</td>
      <td>0.3</td>
      <td>0</td>
      <td>4e-4</td>
      <td>10</td>
      <td>sampled</td>
      <td>all</td>
      <td>—</td>
      <td>0.0593</td>
      <td>больше негативов</td>
    </tr>
    <tr>
      <td>4</td>
      <td>64</td>
      <td>2</td>
      <td>512</td>
      <td>0.1</td>
      <td>8192</td>
      <td>0.05</td>
      <td>0.1</td>
      <td>0</td>
      <td>4e-4</td>
      <td>10</td>
      <td>sampled</td>
      <td>all</td>
      <td>—</td>
      <td>0.0373</td>
      <td>уменьшил fb_weight</td>
    </tr>
    <tr style="background-color: #e8f5e9;">
      <td>5</td>
      <td>64</td>
      <td>2</td>
      <td>512</td>
      <td>0.1</td>
      <td>—</td>
      <td>0.05</td>
      <td>0.3</td>
      <td>0</td>
      <td>4e-4</td>
      <td>10</td>
      <td>full</td>
      <td>all</td>
      <td>—</td>
      <td>0.0890</td>
      <td>заменил sampled softmax на full</td>
    </tr>
    <tr>
      <td>6</td>
      <td>128</td>
      <td>2</td>
      <td>512</td>
      <td>0.1</td>
      <td>—</td>
      <td>0.05</td>
      <td>0.3</td>
      <td>0</td>
      <td>4e-4</td>
      <td>10</td>
      <td>full</td>
      <td>all</td>
      <td>—</td>
      <td>0.0519</td>
      <td>увеличил размер модели</td>
    </tr>
    <tr>
      <td>7</td>
      <td>64</td>
      <td>2</td>
      <td>512</td>
      <td>0.1</td>
      <td>—</td>
      <td>0.05</td>
      <td>0.3</td>
      <td>0</td>
      <td>4e-4</td>
      <td>10</td>
      <td>full</td>
      <td>all</td>
      <td>sliding window</td>
      <td>0.0852</td>
      <td>добавлена аугментация данных</td>
    </tr>
    <tr>
      <td>8</td>
      <td>64</td>
      <td>2</td>
      <td>512</td>
      <td>0.1</td>
      <td>—</td>
      <td>0.05</td>
      <td>0.3</td>
      <td>0</td>
      <td>4e-4</td>
      <td>10</td>
      <td>full</td>
      <td>add_to_cart</td>
      <td>sliding window</td>
      <td>0.0870</td>
      <td>только add_to_cart события</td>
    </tr>
    <tr>
      <td>9</td>
      <td>64</td>
      <td>2</td>
      <td>512</td>
      <td>0.1</td>
      <td>—</td>
      <td>0.05</td>
      <td>0.3</td>
      <td>0</td>
      <td>4e-4</td>
      <td>10</td>
      <td>full</td>
      <td>add_to_cart</td>
      <td>sliding window</td>
      <td>0.0420</td>
      <td>вернул d_model=64</td>
    </tr>
    <tr style="background-color: #e8f5e9;">
      <td>10</td>
      <td>64</td>
      <td>2</td>
      <td>512</td>
      <td>0.1</td>
      <td>—</td>
      <td>1.0</td>
      <td>0.1</td>
      <td>0</td>
      <td>4e-4</td>
      <td>10</td>
      <td>full</td>
      <td>add_to_cart</td>
      <td>sliding window</td>
      <td>0.1048</td>
      <td>убрал температуру</td>
    </tr>
    <tr>
      <td>11</td>
      <td>64</td>
      <td>2</td>
      <td>512</td>
      <td>0.1</td>
      <td>—</td>
      <td>1.0</td>
      <td>0.1</td>
      <td>0.1</td>
      <td>4e-4</td>
      <td>10</td>
      <td>full</td>
      <td>add_to_cart</td>
      <td>sliding window</td>
      <td>0.1026</td>
      <td>label_smoothing=0.1</td>
    </tr>
    <tr style="background-color: #c8e6c9; font-weight: bold;">
      <td>12</td>
      <td>64</td>
      <td>4</td>
      <td>512</td>
      <td>0.2</td>
      <td>—</td>
      <td>1.0</td>
      <td>0.1</td>
      <td>0</td>
      <td>1e-3</td>
      <td>10</td>
      <td>full</td>
      <td>add_to_cart</td>
      <td>sliding window</td>
      <td>0.1268</td>
      <td>увеличил емкость модели</td>
    </tr>
    <tr>
      <td>13</td>
      <td>64</td>
      <td>4</td>
      <td>512</td>
      <td>0.2</td>
      <td>—</td>
      <td>1.0</td>
      <td>0.1</td>
      <td>0</td>
      <td>1e-3</td>
      <td>3</td>
      <td>full</td>
      <td>all</td>
      <td>sliding window</td>
      <td>0.1098</td>
      <td>все события</td>
    </tr>
  </tbody>
</table>


`Наилучшая конфигурация смогла достичь NDCG@100=0.1268.`